In [1]:
pip install fredapi python-dotenv pandas



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from fredapi import Fred
from dotenv import load_dotenv
import pandas as pd
import os

In [7]:
load_dotenv()
FRED_API_KEY = os.getenv("FRED_API_KEY")
fred_auth = fred = Fred(api_key=FRED_API_KEY)

# Point In Time Logic (Predicting on a monthly time unit)

* If your model is **designed to run on the 1st of the month** (e.g., Jan 1st), it has to use whatever the most recently published data is on that exact day.
* When we train our model using historical data, we are not just trying to find historical patterns. We are trying to simulate exactly what the model would have seen if it had been running live on that historical day.

### Example
* When we are training the model, its supposed to predict on a time period of one month (let's say the 31st of each month)
* When it predicts on the 31st of the month you are try to fetch all the data, but for january's unemployment rate, cpi, etc. all of these things aren't available until mid feb
* So actually, you are supposed to use last month's data to do the prediction (e.g. the december data is used for january prediction)

In [48]:
recession_data = pd.DataFrame(fred_auth.get_series("USREC")).reset_index().rename(columns={"index": "date", 0: "value"})

# list of metrics to pull from FRED: consumer price index, payroll employment, unemployment rate, industrial production, 10 year minus 3 month treasury yield
# 'realtime_start' = The Publication Day / Knowledge Date
# 'date' = The Observation Month
# 'value' = The CPI value at that time

monthly_revised = ["CPIAUCSL", "PAYEMS", "UNRATE", "INDPRO"]
daily_market = ["T10Y3M"]

economic_data = {}

for metric in monthly_revised:
    print(f"Pulling first-release data for {metric}...")
    df = fred_auth.get_series_all_releases(metric, realtime_start="2001-01-01")
    # The reason for first releases is because we are assessing the data as it was available at the time of release
    # because this often guides future economic decisions and market reactions.
    # we also only keep the 'realtime_start' and 'value' columns, as we our focusing on the publication day
    first_release_df = df.groupby("date").head(1)
    first_release_df["aligned_date"] = pd.to_datetime(first_release_df["realtime_start"]) + pd.offsets.MonthBegin(1)
    first_release_df.rename(columns={"realtime_start": "publication_date",'date': 'observation_month','value': f"{metric}"}, inplace=True)
    economic_data[metric] = first_release_df


for metric in daily_market:
    print(f"Pulling market data for {metric}...")
    economic_data[metric] = fred_auth.get_series(metric, observation_start="2001-01-01")
    # resample for last day of the month
    # convert to period and back to timestamp to get the last day of the month, 
    # reset index and rename columns
    economic_data[metric] = (
        pd.DataFrame(economic_data[metric])
        .resample("ME")
        .last()
        .reset_index()
        .rename(columns={"index": "date", 0: f"{metric}"})
    )
    economic_data[metric]["aligned_date"] = economic_data[metric]["date"] + pd.offsets.MonthBegin(1)

Pulling first-release data for CPIAUCSL...
Pulling first-release data for PAYEMS...
Pulling first-release data for UNRATE...
Pulling first-release data for INDPRO...
Pulling market data for T10Y3M...


- The model now assumes we are running on the [first of each month]
- We pull the most 'recent' available data for cpi/unemployment date, which is the data published on the month before - which was observing for the month before that
- We then also pull the last known yield curve data excluding the day we are running it
- Which is why we have aligned dates for all the data sources which is offset to the first day of the month after the data was published